# Histórico de Atendimentos: notebook de execução

O código vive no GitHub. Este notebook só prepara o ambiente no Colab.

**Antes de rodar:** troque `REPO_URL` na célula 1 pela URL do seu repositório.
Repositório privado: guarde um token em *Secrets* do Colab (chave `GH_TOKEN`) e use `https://<TOKEN>@github.com/...`, sem imprimir o token.

**Runtime novo:** células 1 a 3, depois 4 e 5.
**Mudou o `models.py` no repositório:** `git pull` (célula 1), `migrate` (célula 2), **reiniciar sessão**, célula 3.

## 1. Drive, Django e código do repositório

In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

REPO_URL = "https://github.com/SEU_USUARIO/atendimentos.git"  # <- troque

drive.mount("/content/drive")
DB_DIR = Path("/content/drive/MyDrive/atendimentos_db")
DB_DIR.mkdir(parents=True, exist_ok=True)
os.environ["ATENDIMENTOS_DB"] = str(DB_DIR / "db.sqlite3")

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "django"], check=True)

BASE = Path("/content/atendimentos")
if (BASE / ".git").exists():
    subprocess.run(["git", "-C", str(BASE), "pull", "--ff-only"], check=True)
else:
    if BASE.exists():  # resto da versão antiga (sem git): preserva em vez de apagar
        BASE.rename("/content/atendimentos_antigo")
    subprocess.run(["git", "clone", REPO_URL, str(BASE)], check=True)

print("Código em:", BASE)
print("Banco em: ", os.environ["ATENDIMENTOS_DB"])

## 2. Migrations

In [ ]:
!cd /content/atendimentos && python manage.py migrate

## 3. Setup do Django (rodar sempre após reiniciar a sessão)

In [ ]:
import os, sys, django
from pathlib import Path

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"   # só no notebook (event loop do Jupyter)
os.environ["ATENDIMENTOS_DB"] = "/content/drive/MyDrive/atendimentos_db/db.sqlite3"
sys.path.insert(0, "/content/atendimentos")
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
django.setup()

from django.conf import settings
from operacoes.models import Colaborador, Demanda, Participacao
from operacoes import selectors as sel

print("Banco:", settings.DATABASES["default"]["NAME"])
print("Contagens:", Colaborador.objects.count(), Demanda.objects.count(), Participacao.objects.count())

## 4. Semeadura de teste (só roda se o banco estiver vazio)

In [ ]:
from datetime import date

if Participacao.objects.exists():
    print("Banco já tem dados, semeadura ignorada.")
else:
    joao, maria, pedro = [Colaborador.objects.get_or_create(nome=n)[0] for n in ("João", "Maria", "Pedro")]
    dados = [
        ("Fiscalização de trânsito", date(2026, 11, 6), [joao, maria]),
        ("Apoio operacional",        date(2026, 11, 6), [joao]),
        ("Organização de equipe",    date(2026, 11, 5), [maria, pedro]),
    ]
    for tipo, data, pessoas in dados:
        d, _ = Demanda.objects.get_or_create(tipo=tipo, data=data)
        for c in pessoas:
            Participacao.objects.get_or_create(demanda=d, colaborador=c)
    print("Semeado:", Colaborador.objects.count(), Demanda.objects.count(), Participacao.objects.count())

## 5. Consultas e testes automatizados

In [ ]:
joao = Colaborador.objects.get(nome="João")
print(list(sel.total_por_colaborador()))
print(list(sel.historico(joao.id)))
print(list(sel.comparar(["João", "Maria"])))

In [ ]:
!cd /content/atendimentos && python manage.py test

## 6. Admin do Django (cadastro por tela)

**6a.** Cria o usuário `admin` no banco do Drive. Rode uma vez; se ele já existir, a mensagem avisa e nada muda.
**6b.** Sobe o servidor e abre o Admin em outra aba. As variáveis de host e CSRF abaixo valem só para este subprocesso no Colab e não ficam no repositório.
Para parar o servidor, rode a célula 6c.

In [ ]:
import getpass, subprocess, sys, os

senha = getpass.getpass("Senha do admin (não aparece na tela): ")
env = {**os.environ, "DJANGO_SUPERUSER_PASSWORD": senha}
r = subprocess.run(
    [sys.executable, "manage.py", "createsuperuser", "--noinput",
     "--username", "admin", "--email", "admin@example.com"],
    cwd="/content/atendimentos", env=env, capture_output=True, text=True,
)
print((r.stdout or r.stderr).strip())

In [ ]:
import subprocess, sys, os, time
from google.colab import output

subprocess.run(["pkill", "-f", "manage.py runserver"])
env = {
    **os.environ,
    "DJANGO_ALLOWED_HOSTS": "*",
    "DJANGO_CSRF_TRUSTED_ORIGINS": "https://*.googleusercontent.com",
}
log = open("/content/runserver.log", "w")
server = subprocess.Popen(
    [sys.executable, "manage.py", "runserver", "8000", "--noreload"],
    cwd="/content/atendimentos", env=env, stdout=log, stderr=log,
)
time.sleep(4)
print("Servidor no ar (pid %s). Log em /content/runserver.log" % server.pid)
output.serve_kernel_port_as_window(8000, path="/admin/")

In [ ]:
!pkill -f "manage.py runserver" && echo "Servidor parado"